# Purpose: Find out how successful flies find the target

In [1]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
from mapd.kinematics import (
    detect_bouts_across_trials, rms_velocity,
    work, positive_effort, holding_cost,
    STATE_REST, STATE_DRIFT, STATE_MOVE,
    _STATE_COLORS, _STATE_LABEL,
)
from mapd.sinq_builders import build_composite_sinq
import h5py
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg as FigureCanvas
import matplotlib.colors as mcolors
import seaborn as sns

import plotly.express as px
from collections import defaultdict
import glob


import pickle
%load_ext autoreload
%autoreload 2
%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

# Single out the learners for this analysis

In [2]:
# sinq2 = Sinq(sinqname='Fig2_learners')
# print(sinq2.__repr__())
# sinq2.df

sinq_0 = Sinq(sinqname='Figure1')
# sinq_1 = Sinq(sinqname='Fig1_control')
# sinq = build_composite_sinq(name='Figure2_learn_v_no_learn_lda',sources=[sinq_0,sinq_1])

# op_df = sinq.display_df(show_tags=True)
# op_df['learn'] = sinq.has_tag('learn')
# op_df['control'] = sinq.has_tag('control')
# op_df[['genotype','tags','learn','control']]

op_df = sinq_0.display_df(show_tags=True)
op_df['learn'] = sinq_0.has_tag('learn')
op_df['control'] = sinq_0.has_tag('control')
# op_df[['genotype','tags','learn','control']]

op_df = op_df.loc[op_df['learn']]
op_df[['genotype','tags','learn','control']]

Found data directory: D:\Data


,genotype,tags,learn,control
210302_F1_C2,Hot-Cell-Gal4 (test),"Figure1, learn",True,False
210405_F1_C1,Hot-Cell-LexA_Chr;81A06_pJFRC7,"Figure1, learn",True,False
210604_F1_C1,Hot-Cell-LexA_Chr;35c09_pJFRC7,"Figure1, learn",True,False
210903_F3_C1,Hot-Cell-LexA_Chr;35C09_pJFRC7,"Figure1, learn",True,False
210915_F1_C1,Hot-Cell-LexA_Chr;78E05_pJFRC7,"Figure1, learn",True,False
210917_F2_C1,Hot-Cell-LexA_Chr;31H05_pJFRC7,"Figure1, learn",True,False
250227_F2_C1,SS61350_pJFRC7,"Figure1, learn",True,False
250227_F3_C1,SS61350_pJFRC7,"Figure1, learn",True,False
250228_F2_C1,SS61350_pJFRC7,"Figure1, learn",True,False
250304_F3_C1,SS61350_pJFRC7,"Figure1, learn",True,False


# Figure 3A - outcomes, how flies solve


In [3]:
# Only do this for new ones

for id in op_df.index:
    folder_path = f'./Figure3/{id}'
    if os.path.exists(folder_path):
        print(f"Folder '{folder_path}' exists. Continuing...")
        continue
    else:
        print(f"Folder '{folder_path}' does not exist.")


    T = sinq_0.restore_table(id)
    T.extract_trial_properties()
    fig,ax = T.plot_outcomes(savefig=True,format='png',fig_dir=f'./Figure2/{id}')

    T.compute_trial_method('prestim_holding_cost')
    fig,ax = T.plot_trial_computations('prestim_holding_cost',savefig=True,fig_dir=f'./Figure2/{id}')

    T.compute_trial_method('prestim_v_rms')
    fig,ax = T.plot_trial_computations('prestim_v_rms',savefig=True,fig_dir=f'./Figure2/{id}')

    T.compute_trial_method('probe_effort')
    fig,ax = T.plot_trial_computations('probe_effort',savefig=True,fig_dir=f'./Figure2/{id}')

    T.compute_trial_method('probe_positive_effort')
    fig,ax = T.plot_trial_computations('probe_positive_effort',savefig=True,fig_dir=f'./Figure2/{id}')

    T.compute_trial_method('probe_rms_velocity')
    fig,ax = T.plot_trial_computations('probe_rms_velocity',savefig=True,fig_dir=f'./Figure2/{id}')
    display(fig)


plt.close('all')

Folder './Figure3/210302_F1_C2' exists. Continuing...
Folder './Figure3/210405_F1_C1' exists. Continuing...
Folder './Figure3/210604_F1_C1' exists. Continuing...
Folder './Figure3/210903_F3_C1' exists. Continuing...
Folder './Figure3/210915_F1_C1' exists. Continuing...
Folder './Figure3/210917_F2_C1' exists. Continuing...
Folder './Figure3/250227_F2_C1' exists. Continuing...
Folder './Figure3/250227_F3_C1' exists. Continuing...
Folder './Figure3/250228_F2_C1' exists. Continuing...
Folder './Figure3/250304_F3_C1' exists. Continuing...
Folder './Figure3/250425_F1_C1' exists. Continuing...
Folder './Figure3/250506_F2_C1' exists. Continuing...
Folder './Figure3/250514_F1_C1' exists. Continuing...
Folder './Figure3/250923_F2_C1' exists. Continuing...
Folder './Figure3/250924_F2_C1' exists. Continuing...
Folder './Figure3/250925_F1_C1' exists. Continuing...


# Functions

In [4]:
import mapd.kinematics as kin

def export_successes(T: Table, fig_dir='./Figure3', v_th=60, v_rest=30):
    """
    Detect movement bouts for each hard-success trial, compute cumulative
    v_rms and effort, save plot_bouts HTML figures, and return a summary DataFrame.
    """
    html_dir = f'{fig_dir}/{T.dfc}/successes_html'
    export_dir = f'{fig_dir}/{T.dfc}/exports'
    os.makedirs(html_dir, exist_ok=True)
    os.makedirs(export_dir, exist_ok=True)

    T.find_successful_trials()
    rows = T.df.loc[T.df['hard_success'], ['Trial', 'success', 'hard_success', 'soft_success', 'pyasState']].copy()
    rows['next_trial'] = T.df['Trial'].shift(-1).loc[rows.index]
    print(f'{len(rows)} hard-success trials')

    records = []
    for t_num in rows.index:
        row = rows.loc[t_num]
        trial1, trial2 = row['Trial'], row['next_trial']

        bouts, states, t, x = kin.detect_bouts_across_trials(
            [trial1, trial2], v_th=v_th, v_rest=v_rest)

        metrics = kin.bout_cumulative_metrics(bouts, t, x, trials=[trial1, trial2])

        fig = kin.plot_bouts(bouts, states, t, x,
                             trials=[trial1, trial2],
                             v_th=v_th, v_rest=v_rest,
                             show_grid=False, show_drift_patches=True,
                             cum_vrms_all_bouts=True)
        fig.write_html(f'{html_dir}/trial_{t_num}.html')

        records.append({
            'trial_number': t_num,
            'pyasState': row['pyasState'],
            'success': row['success'],
            'hard_success': row['hard_success'],
            'soft_success': row['soft_success'],
            'v_rms': metrics['v_rms'],
            'effort': metrics['effort'],
            'n_bouts': len(bouts),
            'bout_duration': bouts[0]['duration'] if bouts else 0.0,
            'started_from_rest': bouts[0]['started_from_rest'] if bouts else None,
            'dfc': T.dfc,
        })

    df = pd.DataFrame(records)
    pkl_path = f'{export_dir}/{T.dfc}_success_vigor.pkl'
    df.to_pickle(pkl_path)
    print(f'Saved {len(df)} rows → {pkl_path}')
    print(f'HTML figures → {html_dir}/')
    return df


def export_sudden_punishment(T: Table, fig_dir='./Figure3', v_th=60, v_rest=30):
    """
    Detect movement bouts for each sudden-punishment trial, compute cumulative
    v_rms and effort (same as export_successes), save plot_bouts HTML figures
    with the preceding trial shown as context, and return a summary DataFrame.

    Sudden punishment: as_off trials preceded by 2+ no_as trials.
    """
    html_dir = f'{fig_dir}/{T.dfc}/sudden_punishment_html'
    export_dir = f'{fig_dir}/{T.dfc}/exports'
    os.makedirs(html_dir, exist_ok=True)
    os.makedirs(export_dir, exist_ok=True)

    T.find_sudden_punishment_trials()
    rows = T.df.loc[T.df['sudden_punishment'], ['Trial', 'pyasState']].copy()
    rows['next_trial'] = T.df['Trial'].shift(-1).loc[rows.index]
    rows['prev_trial'] = T.df['Trial'].shift(1).loc[rows.index]
    print(f'{len(rows)} sudden-punishment trials')

    records = []
    for t_num in rows.index:
        row = rows.loc[t_num]
        trial1, trial2 = row['Trial'], row['next_trial']
        prev_trial = row['prev_trial']

        bouts, states, t, x = kin.detect_bouts_across_trials(
            [trial1, trial2], v_th=v_th, v_rest=v_rest)

        metrics = kin.bout_cumulative_metrics(bouts, t, x, trials=[trial1, trial2])

        fig = kin.plot_bouts(bouts, states, t, x,
                             trials=[trial1, trial2],
                             v_th=v_th, v_rest=v_rest,
                             show_grid=False, show_drift_patches=True,
                             cum_vrms_all_bouts=True,
                             context_trial=prev_trial)
        fig.write_html(f'{html_dir}/trial_{t_num}.html')

        records.append({
            'trial_number': t_num,
            'pyasState': row['pyasState'],
            'v_rms': metrics['v_rms'],
            'effort': metrics['effort'],
            'n_bouts': len(bouts),
            'bout_duration': bouts[0]['duration'] if bouts else 0.0,
            'started_from_rest': bouts[0]['started_from_rest'] if bouts else None,
            'dfc': T.dfc,
        })

    df = pd.DataFrame(records)
    pkl_path = f'{export_dir}/{T.dfc}_sudden_punishment_vigor.pkl'
    df.to_pickle(pkl_path)
    print(f'Saved {len(df)} rows -> {pkl_path}')
    print(f'HTML figures -> {html_dir}/')
    return df


def export_as_off_row(row,v_th=60, v_rest=30,html_dir=None):
    t_num = row.index
    trial1, trial2 = row['Trial'], row['next_trial']
    if trial2 is None:
        return

    bouts, states, t, x = kin.detect_bouts_across_trials(
        [trial1, trial2], v_th=v_th, v_rest=v_rest)

    metrics = kin.bout_cumulative_metrics(bouts, t, x, trials=[trial1, trial2])

    fig = kin.plot_bouts(bouts, states, t, x,
                            trials=[trial1, trial2],
                            v_th=v_th, v_rest=v_rest,
                            show_grid=False, show_drift_patches=True,
                            cum_vrms_all_bouts=True)
    if not html_dir is None:
        fig.write_html(f'{html_dir}/trial_{t_num}.html')

    return metrics, bouts, fig
    

def export_baseline_as_off(T: Table, fig_dir='./Figure3', v_th=60, v_rest=30):
    """
    Detect movement bouts for baseline as_off trials -- those that are neither
    success (fly stays on target after) nor sudden punishment (fly was on target before).
    These represent 'regular' escape movements.
    """
    html_dir = f'{fig_dir}/{T.dfc}/baseline_as_off_html'
    export_dir = f'{fig_dir}/{T.dfc}/exports'
    os.makedirs(html_dir, exist_ok=True)
    os.makedirs(export_dir, exist_ok=True)

    T.find_successful_trials()
    T.find_sudden_punishment_trials()

    as_off = (T.df['as_outcome'] == 'as_off') | (T.df['as_outcome'] == 'as_off_late')
    is_success = T.df.get('success', False)
    is_spun = T.df.get('sudden_punishment', False)
    is_soft_spun = T.df.get('soft_sudden_punishment', False)
    baseline = as_off & ~is_success & ~is_spun & ~is_soft_spun

    rows = T.df.loc[baseline, ['Trial', 'pyasState']].copy()
    rows['next_trial'] = T.df['Trial'].shift(-1).loc[rows.index]
    print(f'{len(rows)} baseline as_off trials')

    records = []
    for t_num in rows.index:
        html_dir = f'{fig_dir}/{T.dfc}/baseline_as_off_html'
        bouts, metrics = export_as_off_row(rows.loc[t_num], html_dir=html_dir)

        records.append({
            'trial_number': t_num,
            'pyasState': rows.loc[t_num,'pyasState'],
            'v_rms': metrics['v_rms'],
            'effort': metrics['effort'],
            'n_bouts': len(bouts),
            'bout_duration': bouts[0]['duration'] if bouts else 0.0,
            'started_from_rest': bouts[0]['started_from_rest'] if bouts else None,
            'dfc': T.dfc,
        })

    df = pd.DataFrame(records)
    pkl_path = f'{export_dir}/{T.dfc}_baseline_as_off_vigor.pkl'
    df.to_pickle(pkl_path)
    print(f'Saved {len(df)} rows -> {pkl_path}')
    print(f'HTML figures -> {html_dir}/')
    return df

In [5]:
# Export all trial types in one pass
# for dfc in op_df.index:
#     if not op_df.loc[dfc, 'learn']:
#         continue
#     T = sinq_0.restore_table(dfc)
#     T.extract_trial_properties()
#     export_successes(T)
#     export_sudden_punishment(T)
#     export_baseline_as_off(T)
#     sinq_0.drop_tables()

In [6]:
T = sinq_0.restore_table(dayflycell='210302_F1_C2')

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\210302\\210302_F1_C2\\LEDFlashWithPiezoCueControl_210302_F1_C2_Table.parquet")
Getting trials
Excluding trials:
[Trial(trial=239, 210302_F1_C2, dT=10.44668, ex=True), Trial(trial=355, 210302_F1_C2, dT=14.47674, ex=True), Trial(trial=474, 210302_F1_C2, dT=8.9779, ex=True), Trial(trial=602, 210302_F1_C2, dT=8.9568, ex=True)]
Getting all meta keys
['as_outcome', 'fiberLED', 'filterLED', 'filtercube_status', 'probeZero', 'pyasState', 'pyasWidth', 'pyasXPosition']
Found 2 target positions - Counter({('lo', 650.0, -180.0, 80.0): 403, ('hi', 650.0, -280.0, 80.0): 397})


In [7]:
# export_as_off_row()
T.df.columns
row = T.df.loc[507].copy()
row['next_trial'] = T.df.loc[507+1,'Trial']
_,_,fig = export_as_off_row(row)
fig.show()

# Success vigor metrics

In [8]:
# Load all success vigor exports
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

vigor_pkls = glob.glob('./Figure3/*/exports/*_success_vigor.pkl')
all_vigor = pd.concat([pd.read_pickle(p) for p in vigor_pkls], ignore_index=True)
print(f'{len(vigor_pkls)} flies, {len(all_vigor)} total hard-success trials')
all_vigor.head()

16 flies, 361 total hard-success trials


,trial_number,pyasState,success,hard_success,soft_success,v_rms,effort,n_bouts,bout_duration,started_from_rest,dfc
0,32,hi,True,True,True,73.172145,4261.253761,1,8.539299,False,210302_F1_C2
1,113,hi,True,True,True,75.445106,3054.792367,3,2.638728,False,210302_F1_C2
2,122,hi,True,True,True,81.473335,2510.580008,1,3.795130,False,210302_F1_C2
3,144,hi,True,True,True,71.979328,3432.466093,1,5.320011,False,210302_F1_C2
4,155,lo,True,True,True,0.000000,0.000000,0,0.000000,None,210302_F1_C2


In [9]:
# v_rms vs effort with marginal ECDFs, colored by fly
import matplotlib.colors as mcolors
import seaborn as sns

cats = sorted(all_vigor['dfc'].unique())
palette = sns.color_palette('Set2', len(cats))
color_map_hex = {g: mcolors.to_hex(c) for g, c in zip(cats, palette)}

fig = make_subplots(
    rows=2, cols=2,
    column_widths=[0.80, 0.20],
    row_heights=[0.20, 0.80],
    specs=[[{"type": "scatter"}, None],
           [{"type": "scatter"}, {"type": "scatter"}]],
    horizontal_spacing=0.05, vertical_spacing=0.05,
)

for cat in cats:
    sel = all_vigor['dfc'] == cat
    fig.add_trace(
        go.Scatter(
            x=all_vigor.loc[sel, 'v_rms'],
            y=all_vigor.loc[sel, 'effort'],
            mode='markers',
            name=str(cat),
            marker=dict(color=color_map_hex[cat], size=5, opacity=0.9),
            hovertext=all_vigor.loc[sel, 'trial_number'],
            hovertemplate=(
                "v_rms=%{x:.1f}<br>effort=%{y:.1f}<br>"
                f"dfc={cat}<br>trial=%{{hovertext}}<extra></extra>"
            ),
        ),
        row=2, col=1,
    )

# Top ECDF (v_rms)
vals = np.sort(all_vigor['v_rms'].dropna().values)
ecdf_y = np.arange(1, len(vals) + 1) / len(vals)
fig.add_trace(go.Scatter(x=vals, y=ecdf_y, mode='lines',
              line=dict(color='black', width=1.5), showlegend=False), row=1, col=1)

# Right ECDF (effort)
vals = np.sort(all_vigor['effort'].dropna().values)
ecdf_x = np.arange(1, len(vals) + 1) / len(vals)
fig.add_trace(go.Scatter(x=ecdf_x, y=vals, mode='lines',
              line=dict(color='black', width=1.5), showlegend=False), row=2, col=2)

fig.add_annotation(text=f'n = {len(all_vigor)} trials, {len(cats)} flies',
                   xref='paper', yref='paper', x=0.01, y=0.99,
                   showarrow=False, font=dict(size=14))

fig.update_layout(
    width=1100, height=850, title='v_rms vs effort (all hard successes)',
    margin=dict(l=60, r=260, t=70, b=50), font=dict(size=16),
    legend_title_text='dfc',
)
fig.update_xaxes(title_text='v_rms (um/s)', row=2, col=1)
fig.update_yaxes(title_text='effort (uN·um)', row=2, col=1)
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(title_text='ECDF', row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=2)
fig.update_xaxes(title_text='ECDF', row=2, col=2)

# fig.update_xaxes(matches='x3', row=1, col=1)
# fig.update_yaxes(matches='y3', row=2, col=2)

# Set shared limits for the marginal histogram plots                                                                    
x_range = [all_vigor['v_rms'].quantile(0.99)*-.05, all_vigor['v_rms'].quantile(0.99) * 1.1]                                                                
y_range = [all_vigor['effort'].quantile(0.99)*-.05, all_vigor['effort'].quantile(0.99) * 1.1]                                                                                                                                                                                       
fig.update_xaxes(range=x_range, row=1, col=1)  # top hist                                                               
fig.update_xaxes(range=x_range, row=2, col=1)  # scatter                                                                
fig.update_yaxes(range=y_range, row=2, col=1)  # scatter                                                                
fig.update_yaxes(range=y_range, row=2, col=2)  # right hist  

fig.write_image('./Figure3/success_v_rms_vs_effort_by_fly.svg')
fig.show()

In [10]:
# Per-fly summary: box plots of v_rms, effort, bout_duration, n_bouts
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['v_rms (um/s)', 'effort (uN·um)',
                                    'bout_duration (s)', 'n_bouts'])

for i, col in enumerate(['v_rms', 'effort', 'bout_duration', 'n_bouts']):
    row, c = divmod(i, 2)
    for dfc in sorted(all_vigor['dfc'].unique()):
        vals = all_vigor.loc[all_vigor['dfc'] == dfc, col]
        fig.add_trace(
            go.Box(y=vals, name=dfc, legendgroup=dfc,
                   showlegend=(i == 0), marker=dict(size=4)),
            row=row + 1, col=c + 1,
        )

fig.update_layout(height=800, width=1200, title='Success metrics by fly',
                  showlegend=True)
fig.show()

In [11]:
# v_rms vs effort with marginal ECDFs, colored by pyasState
state_colors = {'hi': '#1f77b4', 'lo': '#d62728'}
state_cats = [s for s in ['hi', 'lo'] if s in all_vigor['pyasState'].unique()]

fig = make_subplots(
    rows=2, cols=2,
    column_widths=[0.80, 0.20],
    row_heights=[0.20, 0.80],
    specs=[[{"type": "scatter"}, None],
           [{"type": "scatter"}, {"type": "scatter"}]],
    horizontal_spacing=0.05, vertical_spacing=0.05,
)

for st in state_cats:
    sel = all_vigor['pyasState'] == st
    n_st = sel.sum()
    fig.add_trace(
        go.Scatter(
            x=all_vigor.loc[sel, 'v_rms'],
            y=all_vigor.loc[sel, 'effort'],
            mode='markers',
            name=f'{st} (n={n_st})',
            marker=dict(color=state_colors[st], size=6, opacity=0.8),
            hovertext=all_vigor.loc[sel, 'trial_number'].astype(str) + ' ' + all_vigor.loc[sel, 'dfc'],
            hovertemplate="v_rms=%{x:.1f}<br>effort=%{y:.1f}<br>%{hovertext}<extra></extra>",
        ),
        row=2, col=1,
    )

for st in state_cats:
    sel = all_vigor['pyasState'] == st
    # Top ECDF
    vals = np.sort(all_vigor.loc[sel, 'v_rms'].dropna().values)
    ecdf_y = np.arange(1, len(vals) + 1) / len(vals)
    fig.add_trace(go.Scatter(x=vals, y=ecdf_y, mode='lines',
                  line=dict(color=state_colors[st], width=1.5), showlegend=False), row=1, col=1)
    # Right ECDF
    vals = np.sort(all_vigor.loc[sel, 'effort'].dropna().values)
    ecdf_x = np.arange(1, len(vals) + 1) / len(vals)
    fig.add_trace(go.Scatter(x=ecdf_x, y=vals, mode='lines',
                  line=dict(color=state_colors[st], width=1.5), showlegend=False), row=2, col=2)

fig.add_annotation(text=f'n = {len(all_vigor)} trials',
                   xref='paper', yref='paper', x=0.01, y=0.99,
                   showarrow=False, font=dict(size=14))

fig.update_layout(
    width=1100, height=850, title='v_rms vs effort by target state (hi vs lo)',
    margin=dict(l=60, r=260, t=70, b=50), font=dict(size=16),
)
fig.update_xaxes(title_text='v_rms (um/s)', row=2, col=1)
fig.update_yaxes(title_text='effort (uN·um)', row=2, col=1)
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(title_text='ECDF', row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=2)
fig.update_xaxes(title_text='ECDF', row=2, col=2)

# Set shared limits for the marginal histogram plots                                                                    
x_range = [all_vigor['v_rms'].quantile(0.99)*-.05, all_vigor['v_rms'].quantile(0.99) * 1.1]                                                                
y_range = [all_vigor['effort'].quantile(0.99)*-.05, all_vigor['effort'].quantile(0.99) * 1.1]                                                                                                                                                                                       
fig.update_xaxes(range=x_range, row=1, col=1)  # top hist                                                               
fig.update_xaxes(range=x_range, row=2, col=1)  # scatter                                                                
fig.update_yaxes(range=y_range, row=2, col=1)  # scatter                                                                
fig.update_yaxes(range=y_range, row=2, col=2)  # right hist  

fig.write_image('./Figure3/success_v_rms_vs_effort_by_state.svg')
fig.show()

# Analyze the vigor of "sudden punishment": 
Sudden punishment is defined as as_off trials that follow at least two no_as trials (no_as+no_as+as_off). These are trials where the fly has been holding probe in the target before suddenly getting punished. They are a reversed version of success (as_off + no_as + no_as).

# Compare success vs sudden punishment vs baseline as_off

In [12]:
# Load all three trial-type exports
success_pkls = glob.glob('./Figure3/*/exports/*_success_vigor.pkl')
spun_pkls = glob.glob('./Figure3/*/exports/*_sudden_punishment_vigor.pkl')
baseline_pkls = glob.glob('./Figure3/*/exports/*_baseline_as_off_vigor.pkl')

all_success = pd.concat([pd.read_pickle(p) for p in success_pkls], ignore_index=True)
all_success['trial_type'] = 'success'

all_spun = pd.concat([pd.read_pickle(p) for p in spun_pkls], ignore_index=True)
all_spun['trial_type'] = 'sudden_punishment'

all_baseline = pd.concat([pd.read_pickle(p) for p in baseline_pkls], ignore_index=True)
all_baseline['trial_type'] = 'baseline_as_off'

combined = pd.concat([all_success, all_spun, all_baseline], ignore_index=True)
print(f'Success: {len(all_success)}, Sudden punishment: {len(all_spun)}, '
      f'Baseline as_off: {len(all_baseline)}, Total: {len(combined)}')
combined.head()

Success: 361, Sudden punishment: 361, Baseline as_off: 1877, Total: 2599


,trial_number,pyasState,success,hard_success,soft_success,v_rms,effort,n_bouts,bout_duration,started_from_rest,dfc,trial_type
0,32,hi,True,True,True,73.172145,4261.253761,1,8.539299,False,210302_F1_C2,success
1,113,hi,True,True,True,75.445106,3054.792367,3,2.638728,False,210302_F1_C2,success
2,122,hi,True,True,True,81.473335,2510.580008,1,3.795130,False,210302_F1_C2,success
3,144,hi,True,True,True,71.979328,3432.466093,1,5.320011,False,210302_F1_C2,success
4,155,lo,True,True,True,0.000000,0.000000,0,0.000000,None,210302_F1_C2,success


In [13]:
# v_rms vs effort: 3-way comparison with marginal ECDFs
type_colors = {'success': '#2ca02c', 'sudden_punishment': '#d62728', 'baseline_as_off': '#7f7f7f'}

fig = make_subplots(
    rows=2, cols=2,
    column_widths=[0.80, 0.20],
    row_heights=[0.20, 0.80],
    specs=[[{"type": "scatter"}, None],
           [{"type": "scatter"}, {"type": "scatter"}]],
    horizontal_spacing=0.05, vertical_spacing=0.05,
)

counts = {}
for tt in ['baseline_as_off', 'sudden_punishment', 'success']:
    sel = combined['trial_type'] == tt
    n_tt = sel.sum()
    counts[tt] = n_tt
    fig.add_trace(
        go.Scatter(
            x=combined.loc[sel, 'v_rms'],
            y=combined.loc[sel, 'effort'],
            mode='markers',
            name=f'{tt} (n={n_tt})',
            marker=dict(color=type_colors[tt], size=6, opacity=0.7),
            hovertext=combined.loc[sel, 'trial_number'].astype(str) + ' ' + combined.loc[sel, 'dfc'],
            hovertemplate="v_rms=%{x:.1f}<br>effort=%{y:.1f}<br>%{hovertext}<extra></extra>",
        ),
        row=2, col=1,
    )

for tt in ['baseline_as_off', 'sudden_punishment', 'success']:
    sel = combined['trial_type'] == tt
    # Top ECDF
    vals = np.sort(combined.loc[sel, 'v_rms'].dropna().values)
    ecdf_y = np.arange(1, len(vals) + 1) / len(vals)
    fig.add_trace(go.Scatter(x=vals, y=ecdf_y, mode='lines',
                  line=dict(color=type_colors[tt], width=1.5), showlegend=False), row=1, col=1)
    # Right ECDF
    vals = np.sort(combined.loc[sel, 'effort'].dropna().values)
    ecdf_x = np.arange(1, len(vals) + 1) / len(vals)
    fig.add_trace(go.Scatter(x=ecdf_x, y=vals, mode='lines',
                  line=dict(color=type_colors[tt], width=1.5), showlegend=False), row=2, col=2)

fig.add_annotation(text=f'Total: {len(combined)} trials',
                   xref='paper', yref='paper', x=0.01, y=0.99,
                   showarrow=False, font=dict(size=14))

fig.update_layout(
    width=1100, height=850,
    title='Success vs Sudden Punishment vs Baseline: v_rms vs effort',
    margin=dict(l=60, r=260, t=70, b=50), font=dict(size=16),
)
fig.update_xaxes(title_text='v_rms (um/s)', row=2, col=1)
fig.update_yaxes(title_text='effort (uN\u00b7um)', row=2, col=1)
fig.update_xaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(title_text='ECDF', row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=2)
fig.update_xaxes(title_text='ECDF', row=2, col=2)

# Set shared limits for the marginal histogram plots                                                                    
x_range = [combined['v_rms'].quantile(0.99)*-.05, combined['v_rms'].quantile(0.99) * 1.1]                                                                
y_range = [combined['effort'].quantile(0.99)*-.05, combined['effort'].quantile(0.99) * 1.1]                                                                                                                                                                                       
fig.update_xaxes(range=x_range, row=1, col=1)  # top hist                                                               
fig.update_xaxes(range=x_range, row=2, col=1)  # scatter                                                                
fig.update_yaxes(range=y_range, row=2, col=1)  # scatter                                                                
fig.update_yaxes(range=y_range, row=2, col=2)  # right hist  

fig.write_image('./Figure3/success_vs_spun_vs_baseline_vigor.svg')
fig.show()

In [14]:
# Side-by-side box plots: 3-way comparison
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=['v_rms (um/s)', 'effort (uN\u00b7um)',
                                    'bout_duration (s)', 'n_bouts'])

for i, col in enumerate(['v_rms', 'effort', 'bout_duration', 'n_bouts']):
    row, c = divmod(i, 2)
    for tt in ['success', 'sudden_punishment', 'baseline_as_off']:
        vals = combined.loc[combined['trial_type'] == tt, col]
        fig.add_trace(
            go.Box(y=vals, name=tt, legendgroup=tt,
                   showlegend=(i == 0),
                   marker=dict(size=4, color=type_colors[tt])),
            row=row + 1, col=c + 1,
        )

fig.update_layout(height=700, width=900,
                  title='Success vs Sudden Punishment vs Baseline: metric distributions')
fig.show()

In [15]:
# Mann-Whitney U tests: pairwise comparisons across trial types
from scipy.stats import mannwhitneyu

pairs = [
    ('success', 'baseline_as_off'),
    ('sudden_punishment', 'baseline_as_off'),
    ('success', 'sudden_punishment'),
]

results = []
for metric in ['v_rms', 'effort', 'bout_duration', 'n_bouts']:
    for a_name, b_name in pairs:
        a = combined.loc[combined['trial_type'] == a_name, metric].dropna()
        b = combined.loc[combined['trial_type'] == b_name, metric].dropna()
        stat, p = mannwhitneyu(a, b, alternative='two-sided')
        results.append({
            'metric': metric,
            'comparison': f'{a_name} vs {b_name}',
            'n_a': len(a), 'n_b': len(b),
            'U': stat, 'p': p,
            'median_a': a.median(), 'median_b': b.median(),
        })

stats_df = pd.DataFrame(results)
stats_df['sig'] = stats_df['p'].apply(
    lambda p: '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns')
print(stats_df.to_string(index=False))

       metric                           comparison  n_a  n_b        U            p    median_a     median_b sig
        v_rms           success vs baseline_as_off  361 1877 230029.0 3.905621e-22  325.698025   637.841625 ***
        v_rms sudden_punishment vs baseline_as_off  361 1877 212889.5 4.156131e-29  215.179037   637.841625 ***
        v_rms         success vs sudden_punishment  361  361  71910.0 1.594192e-02  325.698025   215.179037   *
       effort           success vs baseline_as_off  361 1877 218201.0 7.726767e-27 6967.829820 17267.979281 ***
       effort sudden_punishment vs baseline_as_off  361 1877 211424.5 9.470497e-30 6344.247074 17267.979281 ***
       effort         success vs sudden_punishment  361  361  69979.0 8.531754e-02 6967.829820  6344.247074  ns
bout_duration           success vs baseline_as_off  361 1877 278353.0 7.626423e-08    4.055006     4.876830 ***
bout_duration sudden_punishment vs baseline_as_off  361 1877 281374.0 3.271000e-07    4.218489     4.876

In [ ]:
# Multivariate tests on (v_rms, effort) jointly
from skbio.stats.distance import permanova, DistanceMatrix
from scipy.spatial.distance import pdist, squareform
from dcor.homogeneity import energy_test

metrics = ['v_rms', 'effort']
mv = combined[metrics + ['trial_type']].dropna()

# Standardize so both metrics contribute equally to distances
from sklearn.preprocessing import StandardScaler
X_std = StandardScaler().fit_transform(mv[metrics])

# --- PERMANOVA: omnibus test across all 3 groups ---
dm = DistanceMatrix(squareform(pdist(X_std, 'euclidean')))
result = permanova(dm, mv['trial_type'].values, permutations=9999)
print('=== PERMANOVA (all 3 groups) ===')
print(f"  pseudo-F = {result['test statistic']:.3f},  "
      f"p = {result['p-value']:.4f},  n = {len(mv)}")
print()

# --- Pairwise energy distance tests ---
print('=== Energy distance tests (pairwise, 9999 permutations) ===')
for a_name, b_name in pairs:
    xa = X_std[mv['trial_type'].values == a_name]
    xb = X_std[mv['trial_type'].values == b_name]
    stat, p_val = energy_test(xa, xb, num_resamples=9999)
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f'  {a_name} vs {b_name}: E = {stat:.4f}, p = {p_val:.4f} {sig}')